In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


PyTorch version: 2.8.0+cu126
CUDA available: True
Device name: Tesla T4


In [ ]:
import torch
import transformers

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


from transformers import AutoTokenizer, AutoModel

model_name = "microsoft/deberta-v3-base"

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n✅ Successfully loaded {model_name}")
except Exception as e:
    print("\n❌ Error loading model:", e)


PyTorch version: 2.8.0+cu126
Transformers version: 4.57.1
CUDA available: True
Device: Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]


✅ Successfully loaded microsoft/deberta-v3-base


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving docs_sample.json to docs_sample (1).json


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving questions.jsonl to questions.jsonl


In [ ]:
import json
import os


for name in os.listdir("/content"):
    if name.startswith("docs_sample") and name.endswith(".json"):
        file_path = f"/content/{name}"
        print("✅ Found file:", file_path)
        break
else:
    raise FileNotFoundError("No docs_sample*.json file found in /content")


with open(file_path, "r") as f:
    data = json.load(f)

print(f"Loaded {len(data)} records.")


✅ Found file: /content/docs_sample.json
Loaded 10 records.


In [ ]:
import json
from collections import Counter

def inspect_json_file(file_path, file_type="json"):
    print(f"\n=== Inspecting {file_path} ===")

    items = []
    if file_type == "json":
        with open(file_path, "r", encoding="utf-8") as f:
            items = json.load(f)
    elif file_type == "jsonl":
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                items.append(json.loads(line))

    print(f"Top-level type: {type(items)}")
    print(f"Number of items: {len(items) if isinstance(items, list) else 'N/A'}")


    if isinstance(items, list) and all(isinstance(i, dict) for i in items):
        all_keys = [key for item in items for key in item.keys()]
        key_counts = Counter(all_keys)
        print("Repeated field names / headlines and counts:")
        for key, count in key_counts.most_common():
            print(f"- {key} : appears {count} times")
    else:
        print("Items are not dicts or structure is irregular. Could be text paragraphs or mixed types.")


inspect_json_file("docs_sample.json", file_type="json")

inspect_json_file("questions.jsonl", file_type="jsonl")



=== Inspecting docs_sample.json ===
Top-level type: <class 'list'>
Number of items: 10
Repeated field names / headlines and counts:
- topic_id : appears 10 times
- topic : appears 10 times
- docs : appears 10 times

=== Inspecting questions.jsonl ===
Top-level type: <class 'list'>
Number of items: 200
Repeated field names / headlines and counts:
- topic_id : appears 200 times
- uuid : appears 200 times
- target_event : appears 200 times
- option_A : appears 200 times
- option_B : appears 200 times
- option_C : appears 200 times
- option_D : appears 200 times
- golden_answer : appears 200 times


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import json


with open("questions.jsonl", "r") as f:
    questions = [json.loads(line) for line in f]

model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)

predictions = []

for q in questions:
    question_text = q["target_event"]
    options = [q["option_A"], q["option_B"], q["option_C"], q["option_D"]]

    scores = []
    for opt in options:
        inputs = tokenizer(question_text, opt, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            logits = model(**inputs).logits
            scores.append(logits.item())

    best_idx = int(torch.argmax(torch.tensor(scores)))
    predictions.append({
        "uuid": q["uuid"],
        "predicted_answer": ["A","B","C","D"][best_idx],
        "golden_answer": q["golden_answer"]
    })


for p in predictions[:5]:
    print(p)


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'uuid': '02777771-93de-450d-9a16-b08b25f2e2c8', 'predicted_answer': 'C', 'golden_answer': 'D'}
{'uuid': '42359370-0cc8-475d-bb3e-90623f86e5df', 'predicted_answer': 'C', 'golden_answer': 'C'}
{'uuid': 'a82409ed-dddc-4779-81a7-83a872685c10', 'predicted_answer': 'C', 'golden_answer': 'B'}
{'uuid': '646e4473-68a8-4b6a-8e5a-c788c717f856', 'predicted_answer': 'D', 'golden_answer': 'D'}
{'uuid': '4778d001-0f67-4d20-bf62-7082caa1619b', 'predicted_answer': 'C', 'golden_answer': 'C'}
